In [ ]:
# # Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# import pandas as pd
# data = pd.read_csv('/content/drive/MyDrive/cicids2017_combined.csv')
# # Upload zip files
# from google.colab import files
# uploaded = files.upload()  # 다운받은 zip 파일 선택
# # Unzip and check files
# !unzip -q *.zip -d data/
# import os
# for f in os.listdir('data/'):
#     print(f)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# import pandas as pd
# import numpy as np

# data = pd.read_csv('/content/drive/MyDrive/cicids2017_combined.csv')
# print(f"Loaded: {data.shape}")

In [10]:
from google.colab import files
uploaded = files.upload()  # 다운받은 csv 선택

import pandas as pd
import numpy as np
data = pd.read_csv('cicids2017_combined.csv')
print(f"Loaded: {data.shape}")

KeyboardInterrupt: 

In [ ]:
# EDA
# Load data
import pandas as pd
import os

dfs = []
for f in sorted(os.listdir('data/')):
    if f.endswith('.csv'):
        df = pd.read_csv(f'data/{f}')
        dfs.append(df)
        print(f"{f}: {df.shape}")

data = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {data.shape}")

In [ ]:
# Check label distribuation
print(data[' Label'].value_counts())

In [ ]:
# Visualise attack patterns
import matplotlib.pyplot as plt

label_counts = data[' Label'].value_counts()
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar')
plt.title('Attack Type Distribution in CICIDS2017')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Basic data shapes
print(f"Columns: {data.shape[1]}")
print(f"\nMissing values:\n{data.isnull().sum()[data.isnull().sum() > 0]}")
print(f"\nData types:\n{data.dtypes.value_counts()}")

In [ ]:
# Preprocessing
data.columns = data.columns.str.strip()
print(data.columns.tolist())

In [ ]:
import numpy as np

# Replace infinite values with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check missing values
missing = data.isnull().sum()
print(f"Columns with missing values:\n{missing[missing > 0]}")

# Drop rows with missing values
before = len(data)
data.dropna(inplace=True)
print(f"\nRows removed: {before - len(data)}")
print(f"Remaining data: {data.shape}")

In [ ]:
# Labeling
print(data['Label'].value_counts())

# Create binary label: BENIGN=0, Attack=1
data['is_attack'] = (data['Label'] != 'BENIGN').astype(int)
print(f"\nBenign: {(data['is_attack']==0).sum()}")
print(f"Attack: {(data['is_attack']==1).sum()}")

In [ ]:
feature_cols = data.select_dtypes(include=[np.number]).columns.tolist()
feature_cols.remove('is_attack')
print(f"Number of features: {len(feature_cols)}")

X = data[feature_cols]
y = data['is_attack']
print(f"X: {X.shape}, y: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"Train attack ratio: {y_train.mean():.3f}")
print(f"Test attack ratio: {y_test.mean():.3f}")

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# Train Isolation Forest (unsupervised - only learns "normal")
# Train on benign data only
X_train_benign = X_train_scaled[y_train == 0]

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.2,
    random_state=42
)
iso_forest.fit(X_train_benign)

# Predict on test set (-1 = anomaly, 1 = normal)
y_pred_raw = iso_forest.predict(X_test_scaled)

# Convert: -1 → 1 (attack), 1 → 0 (benign)
y_pred = (y_pred_raw == -1).astype(int)

# Results
print("Isolation Forest Results:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train_scaled, y_train)

y_pred_xgb = xgb.predict(X_test_scaled)

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb, target_names=['Benign', 'Attack']))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_xgb)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compare models
models = ['Isolation Forest', 'XGBoost']
precision = [0.36, 1.00]
recall = [0.47, 1.00]
f1 = [0.41, 1.00]

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, precision, width, label='Precision')
ax.bar(x, recall, width, label='Recall')
ax.bar(x + width, f1, width, label='F1-Score')

ax.set_ylabel('Score')
ax.set_title('Attack Detection: Isolation Forest vs XGBoost')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
# EWMA-based anomaly detection
# Simulates per-source packet rate monitoring

from sklearn.metrics import classification_report, confusion_matrix

# Use Flow Duration and packet counts as rate features
rate_features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                 'Flow Bytes/s', 'Flow Packets/s']

# EWMA on test set
ewma_scores = pd.DataFrame(X_test[rate_features]).ewm(span=20).mean()
residuals = np.abs(X_test[rate_features].values - ewma_scores.values)
anomaly_score = residuals.mean(axis=1)

# Threshold: mean + 2*std
threshold = anomaly_score.mean() + 2 * anomaly_score.std()
y_pred_ewma = (anomaly_score > threshold).astype(int)

print("EWMA Results:")
print(classification_report(y_test, y_pred_ewma, target_names=['Benign', 'Attack']))